## NAME: Souri Rishik Volety
## Reg No: 230968004

### Exercise 1: Hill-Climbing Search for Feature Selection in Predictive Modeling

Problem Statement: You are given a dataset (select a random dataset) with multiple input
features and a target variable. Each state represents a subset of selected features. The goal is to
find a feature subset that maximizes model accuracy.
State Representation: A binary vector indicating whether a feature is selected (1) or not (0)
Initial State: A randomly generated feature subset
Neighbor Generation: Flip one bit (add/remove one feature)
Evaluation Function: Cross-validation accuracy of a simple classifier (e.g., Logistic
Regression)
Algorithm Implementation
• Implement steepest-ascent hill climbing
• Stop when no neighbor improves the current solution
Experimentation: Run the algorithm multiple times with different random initial states

In [5]:
import numpy as np
import random
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

data = load_breast_cancer()
X = data.data
y = data.target

n_features = X.shape[1]

def evaluate(state):
    if sum(state) == 0:
        return 0
    selected_indices = [i for i in range(len(state)) if state[i] == 1]
    X_selected = X[:, selected_indices]
    model = LogisticRegression(max_iter=1000, solver='liblinear')
    scores = cross_val_score(model, X_selected, y, cv=5)
    return scores.mean()

def generate_neighbors(state):
    neighbors = []
    for i in range(len(state)):
        neighbor = state.copy()
        neighbor[i] = 1 - neighbor[i]
        neighbors.append(neighbor)
    return neighbors

def hill_climbing():
    current_state = [random.randint(0, 1) for _ in range(n_features)]
    current_score = evaluate(current_state)
    while True:
        neighbors = generate_neighbors(current_state)
        best_score = current_score
        best_neighbor = current_state
        for neighbor in neighbors:
            score = evaluate(neighbor)
            if score > best_score:
                best_score = score
                best_neighbor = neighbor
        if best_score > current_score:
            current_state = best_neighbor
            current_score = best_score
        else:
            break
    return current_state, current_score

runs = 20

for i in range(runs):
    state, score = hill_climbing()
    print(f"Run {i+1}:")
    print("Selected feature count:", sum(state))
    print("Accuracy:", round(score, 4))
    print()

Run 1:
Selected feature count: 18
Accuracy: 0.9561

Run 2:
Selected feature count: 17
Accuracy: 0.9561

Run 3:
Selected feature count: 14
Accuracy: 0.9596

Run 4:
Selected feature count: 10
Accuracy: 0.9525

Run 5:
Selected feature count: 12
Accuracy: 0.9561

Run 6:
Selected feature count: 16
Accuracy: 0.9543

Run 7:
Selected feature count: 18
Accuracy: 0.9631

Run 8:
Selected feature count: 13
Accuracy: 0.9561

Run 9:
Selected feature count: 14
Accuracy: 0.9508

Run 10:
Selected feature count: 17
Accuracy: 0.9613

Run 11:
Selected feature count: 17
Accuracy: 0.9561

Run 12:
Selected feature count: 15
Accuracy: 0.9578

Run 13:
Selected feature count: 11
Accuracy: 0.9526

Run 14:
Selected feature count: 14
Accuracy: 0.9578

Run 15:
Selected feature count: 13
Accuracy: 0.9596

Run 16:
Selected feature count: 11
Accuracy: 0.9561

Run 17:
Selected feature count: 17
Accuracy: 0.9578

Run 18:
Selected feature count: 15
Accuracy: 0.9561

Run 19:
Selected feature count: 15
Accuracy: 0.9561

Ru

In [6]:
import numpy as np
import random
import math

n_locations = 10
locations = np.random.rand(n_locations, 2) * 100

def total_distance(route):
    dist = 0
    for i in range(len(route)):
        start = locations[route[i]]
        end = locations[route[(i + 1) % len(route)]]
        dist += np.linalg.norm(start - end)
    return dist

def generate_neighbor(route):
    a, b = random.sample(range(len(route)), 2)
    neighbor = route.copy()
    neighbor[a], neighbor[b] = neighbor[b], neighbor[a]
    return neighbor

def simulated_annealing(max_iter=1000, T_init=100, alpha=0.995):
    current_route = list(range(n_locations))
    random.shuffle(current_route)
    current_cost = total_distance(current_route)
    best_route = current_route.copy()
    best_cost = current_cost
    T = T_init
    for i in range(max_iter):
        neighbor = generate_neighbor(current_route)
        neighbor_cost = total_distance(neighbor)
        delta = neighbor_cost - current_cost
        if delta < 0 or random.random() < math.exp(-delta / T):
            current_route = neighbor
            current_cost = neighbor_cost
            if current_cost < best_cost:
                best_route = current_route.copy()
                best_cost = current_cost
        T *= alpha
    return best_route, best_cost

route, cost = simulated_annealing()
print("Best route:", route)
print("Total distance:", round(cost, 2))

Best route: [9, 4, 0, 6, 2, 1, 5, 8, 3, 7]
Total distance: 212.65


In [7]:
import numpy as np
import random

width, height = 100, 100
n_waypoints = 10
population_size = 30
generations = 100
mutation_rate = 0.2
tournament_size = 3
start = np.array([0, 0])
goal = np.array([100, 100])
obstacles = [
    (np.array([40, 50]), 10),
    (np.array([70, 70]), 8),
]

def distance(p1, p2):
    return np.linalg.norm(p1 - p2)

def path_length(path):
    dist = distance(start, path[0])
    for i in range(len(path) - 1):
        dist += distance(path[i], path[i + 1])
    dist += distance(path[-1], goal)
    return dist

def collision_penalty(path):
    penalty = 0
    for wp in path:
        for center, radius in obstacles:
            if distance(wp, center) < radius:
                penalty += 1000
    return penalty

def fitness(path):
    return - (path_length(path) + collision_penalty(path))

def random_path():
    return [np.array([random.uniform(0, width), random.uniform(0, height)]) for _ in range(n_waypoints)]

def tournament_selection(pop, fitnesses):
    best = None
    for _ in range(tournament_size):
        i = random.randrange(len(pop))
        if best is None or fitnesses[i] > fitnesses[best]:
            best = i
    return pop[best]

def one_point_crossover(p1, p2):
    point = random.randint(1, n_waypoints - 1)
    child = p1[:point] + p2[point:]
    return child

def mutate(path):
    new_path = []
    for wp in path:
        if random.random() < mutation_rate:
            wp = wp + np.random.uniform(-5, 5, size=2)
            wp[0] = np.clip(wp[0], 0, width)
            wp[1] = np.clip(wp[1], 0, height)
        new_path.append(wp)
    return new_path

population = [random_path() for _ in range(population_size)]

for gen in range(generations):
    fitnesses = [fitness(p) for p in population]
    new_population = []
    for _ in range(population_size):
        parent1 = tournament_selection(population, fitnesses)
        parent2 = tournament_selection(population, fitnesses)
        child = one_point_crossover(parent1, parent2)
        child = mutate(child)
        new_population.append(child)
    population = new_population

fitnesses = [fitness(p) for p in population]
best_index = np.argmax(fitnesses)
best_path = population[best_index]

print("Best path:")
for wp in best_path:
    print(wp)
print("Total cost:", -fitnesses[best_index])

Best path:
[ 3.67330371 15.03608537]
[ 4.86340077 20.19119503]
[12.59407911 32.53437422]
[15.94189104 35.05302883]
[17.86510781 36.43473967]
[36.38158472 36.53577596]
[40.62802245 35.49527049]
[51.74438844 43.68261981]
[63.98114849 49.98874352]
[86.06442643 73.00954777]
Total cost: 154.62772158009136
